In [3]:
# pip install yfinance pandas numpy

import pandas as pd
import numpy as np
import yfinance as yf

In [4]:
# -----------------------------
# 1) Choose your 30 S&P 500 stocks
# -----------------------------
tickers = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "META",
    "NVDA", "JPM", "V", "PG", "XOM",
    "UNH", "HD", "MA", "COST", "ABBV",
    "PEP", "AVGO", "KO", "MRK", "BAC",
    "CVX", "ADBE", "WMT", "NFLX", "LIN",
    "AMD", "MCD", "TMO", "CSCO", "ORCL"
]

In [5]:
# -----------------------------
# 2) Download 5 years of adjusted-close prices
# -----------------------------
prices = yf.download(
    tickers,
    period="5y",
    interval="1d",
    auto_adjust=False,
    progress=False
)["Adj Close"]

# Clean up
prices = prices.dropna(how="all").sort_index()


In [6]:
# -----------------------------
# 3) Compute daily log returns
# -----------------------------
log_returns = np.log(prices / prices.shift(1)).dropna()

In [8]:
# -----------------------------
# 4) Download VIX from FRED
# -----------------------------
vix_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=VIXCLS"
vix = pd.read_csv(vix_url)

# Clean column names
vix.columns = [str(c).strip().replace("\ufeff", "") for c in vix.columns]

print("VIX columns:", vix.columns.tolist())  # optional debug line

# Handle possible column name differences
date_col = None
vix_col = None

for c in vix.columns:
    cl = c.lower()
    if cl in ["date", "observation_date"]:
        date_col = c
    if cl in ["vixcls", "value"]:
        vix_col = c

if date_col is None or vix_col is None:
    raise ValueError(f"Unexpected VIX columns: {vix.columns.tolist()}")

vix[date_col] = pd.to_datetime(vix[date_col])
vix[vix_col] = pd.to_numeric(vix[vix_col], errors="coerce")

vix = (
    vix.rename(columns={date_col: "Date", vix_col: "VIX"})
       .set_index("Date")
       .dropna()
)

VIX columns: ['observation_date', 'VIXCLS']


In [9]:
def label_vix_regime(v):
    if v < 15:
        return "low"
    elif v < 25:
        return "normal"
    else:
        return "crisis"

vix["regime"] = vix["VIX"].apply(label_vix_regime)
regime_map = {"low": 0, "normal": 1, "crisis": 2}
vix["regime_code"] = vix["regime"].map(regime_map)

In [10]:
# -----------------------------
# 6) Merge returns with VIX/regimes
# -----------------------------
data = log_returns.join(vix[["VIX", "regime", "regime_code"]], how="inner")

In [11]:
# -----------------------------
# 7) 80/20 temporal split (no shuffling)
# -----------------------------
split_idx = int(len(data) * 0.8)

train_data = data.iloc[:split_idx].copy()
test_data  = data.iloc[split_idx:].copy()

# Separate pieces if helpful
X_train_returns = train_data[tickers]
X_test_returns  = test_data[tickers]

vix_train = train_data["VIX"]
vix_test  = test_data["VIX"]

regime_train = train_data["regime_code"]
regime_test  = test_data["regime_code"]


In [12]:
# -----------------------------
# 8) Quick checks
# -----------------------------
print("Prices shape:", prices.shape)
print("Returns shape:", log_returns.shape)
print("Merged data shape:", data.shape)
print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)
print("\nRegime counts (train):")
print(train_data["regime"].value_counts())
print("\nRegime counts (test):")
print(test_data["regime"].value_counts())

print("\nHead of merged dataset:")
print(data.head())

Prices shape: (1256, 30)
Returns shape: (1255, 30)
Merged data shape: (1254, 33)
Train shape: (1003, 33)
Test shape: (251, 33)

Regime counts (train):
regime
normal    619
low       236
crisis    148
Name: count, dtype: int64

Regime counts (test):
regime
normal    204
low        25
crisis     22
Name: count, dtype: int64

Head of merged dataset:
                AAPL      ABBV      ADBE       AMD      AMZN      AVGO  \
Date                                                                     
2021-03-18 -0.034493 -0.012164 -0.026580 -0.056127 -0.034963 -0.041029   
2021-03-19 -0.004490 -0.003379  0.005269  0.011961  0.015393  0.029756   
2021-03-22  0.027942  0.023697  0.024411  0.015563  0.011611  0.001727   
2021-03-23 -0.006913 -0.010060  0.017072 -0.024201  0.008524 -0.023890   
2021-03-24 -0.020196 -0.017124 -0.019064 -0.024539 -0.016204 -0.014740   

                 BAC      COST      CSCO       CVX  ...       PEP        PG  \
Date                                                .